# Cats vs dogs with Logistic Regression

## 1. Objective and decision rule
Cat is 0 and Dog is 1. Logistic Regression learns a linear score $z=w^Tx+b$. The Sigmoid converts it into an estimated Dog probability:

$$P(\mathrm{Dog}|x)=\frac{1}{1+e^{-z}}.$$

A probability of at least 0.5 predicts Dog; otherwise we predict Cat.

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd() if (Path.cwd() / 'common.py').exists() else Path.cwd().parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from common import LABELS, SEED, result_summary, prediction_examples, comparison_table
OUT = ROOT / 'outputs/improved'
plt.rcParams.update({'figure.dpi':110,'font.size':10})


## 2. Use better image features
The original baseline flattened 24 × 24 pixels. We retain that model and compare class weighting and 32 × 32 inputs. The larger improvement uses 576 features from a frozen pretrained CNN, MobileNetV3-Small. It supplies image features; Logistic Regression remains the classifier.

Pillow and torchvision apply the extractor's RGB, resize, crop and normalization steps. NumPy stores the features. A scikit-learn Pipeline learns feature scaling and Logistic Regression. pandas reads the same saved split IDs, and Matplotlib shows evaluation figures.

In [ ]:
import joblib
from scipy.special import expit, softmax
from train_improved import extract_features, train_logistic
manifest = pd.read_csv(ROOT / 'data/processed/split_manifest.csv')
features = extract_features()
labels = LABELS[:2]
keep = manifest.label.isin(labels).to_numpy()
manifest = manifest[keep].reset_index(drop=True)
X = features[keep]
y = manifest.label_id.to_numpy()
train = manifest.split.eq('train').to_numpy()
validation = manifest.split.eq('validation').to_numpy()
test = manifest.split.eq('test').to_numpy()


## 3. Train and select Logistic Regression
Three stratified group folds inside the training split compare C = 0.01, 0.1 and 1, with or without balanced class weights. A smaller C gives stronger regularization. Each held-out fold is scored after fitting the scaler only on its training folds. CV macro-F1 chooses the pipeline; validation is checked separately.

Macro-F1 averages the class F1 scores equally. Detailed fold scores and candidate settings are saved in `outputs/improved/`; the main results are shown as percentages below. The test split never chooses C, class weighting or a checkpoint.

In [ ]:
# Reuse this completed experiment unless retraining is requested.
RETRAIN = False
model_path = ROOT / 'models/binary_features.joblib'
if RETRAIN or not model_path.exists():
    train_logistic(binary=True)
model = joblib.load(model_path)
probabilities = model.predict_proba(X[test])
result_summary('binary_features', labels)


## 4. Connect the probabilities to the equation
Dog is the positive class. True positives are Dogs predicted as Dog; true negatives are Cats predicted as Cat. A false positive is a Cat called Dog, and a false negative is a Dog called Cat. Precision measures how often Dog predictions are right; recall measures how many labelled Dogs are found.

In [ ]:
score = model.decision_function(X[test][:1])[0]
dog_probability = expit(score)
np.testing.assert_allclose(dog_probability, probabilities[0,1], rtol=1e-5)
display(Markdown(f'For the first test image, **z = {score:.2f}** gives **P(Dog) = {dog_probability:.1%}**.'))
z = np.linspace(-7,7,250)
fig,ax = plt.subplots(figsize=(5.5,3))
ax.plot(z,expit(z)); ax.axhline(0.5,color='gray',linestyle='--')
ax.scatter([score],[dog_probability],color='crimson')
ax.set(xlabel='Linear score z',ylabel='Estimated Dog probability',title='Sigmoid decision rule')
fig.tight_layout(); plt.show()


## 5. Compare the original and improved inputs
Training and validation scores help reveal overfitting. These comparisons use the same task and test image IDs.

In [ ]:
rows = []
for label,record,folder in [('Original pixels','binary',ROOT / 'outputs'),
                            ('Weighted pixels','binary_weighted_pixels',OUT),
                            ('Pretrained features','binary_features',OUT)]:
    s = json.loads((folder / f'{record}_metrics.json').read_text())['summary']
    rows.append({'Input':label,'Accuracy':f"{s['accuracy']:.2%}",'Macro-F1':f"{s['f1_macro']:.2%}"})
display(pd.DataFrame(rows).set_index('Input'))
behaviour = pd.read_csv(OUT / 'binary_features_behaviour.csv').set_index('split')
display(behaviour.rename(columns={'accuracy':'Accuracy','macro_f1':'Macro-F1'}).map(lambda value:f'{value:.2%}'))


## 6. Inspect real predictions
The examples include correct and incorrect predictions. Labels come from the dataset and can contain errors. Estimated probabilities are not calibrated guarantees.

In [ ]:
prediction_examples('binary_features', labels)

## 7. Evaluation limits and a separate check
The original Animals-10 test split was inspected before this improvement study. Its scores are historical comparisons, not an untouched final assessment.

A separate licensed iNaturalist audit, selected and visually reviewed before inference, gave **64.86% accuracy** and **64.76% macro-F1** on 37 animal photographs. The full 100-observation audit and the photo eligibility decisions are also saved. No external result was used for tuning or sample replacement.

This small sample includes distant, dark and partly hidden animals. It shows a substantial domain gap. Contributor labels and dates, missing original source identities and incomplete duplicate detection limit any independence guarantee. These models are teaching classifiers; their historical scores do not establish reliable performance on arbitrary new photos.